### 🖼️ Dataset Thumbnails (ZennyKenny___tactical-military-reasoning-v.1.0)

![Thumbnail](../thumbnails/ZennyKenny___tactical-military-reasoning-v.1.0_01.png)

In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import time
from typing import Iterator, Dict, Any

# ========================================================================
# 🧠 튜터가 알려주는 데이터셋 정보 (AI 스터디 가이드)
# ========================================================================
# 📖 데이터셋 이름: ZennyKenny/tactical-military-reasoning-v.1.0
# 🌍 한글 제목: 전술 군사 추론 데이터셋 v.1.0 (Tactical Military Reasoning)
# ✨ 데이터셋 의미: 이 데이터셋은 군사 시나리오(전장 상황)를 기반으로,
#             특정 상황에서 아군이 '공격'을 할 때 필요한 논리적인 추론 과정과
#             적군이 '방어'할 때 취해야 할 논리적 대응 전략을 미리 생성해 둔 데이터입니다.
# 🚀 학습 목표: LLM(대규모 언어 모델)에게 '상황'을 주고, 그 상황에 맞는
#               '논리적 추론'을 생성하게 만드는 프롬프트 엔지니어링 실습에 최적화되어 있습니다.
# ========================================================================


# --- 상수 설정 ---
DATASET_NAME = "ZennyKenny/tactical-military-reasoning-v.1.0"
TARGET_SPLIT = 'train'
SAMPLE_COUNT = 5 # 초보자 실습이므로, 5개의 샘플만 보여줄게요!

# ========================================================================
# 🛠️ 1단계: 데이터셋 설정 및 로딩 (스트리밍 vs. 일반 모드 처리)
# ========================================================================

print("🌟 안녕하세요! 오늘은 군사 전략 시나리오를 분석하는 AI 모델을 만들어 볼 거예요! 🤖")
print("이 데이터셋은 단순히 텍스트가 아니라, '상황(Scenario)'과 '전략적 판단(Reasoning)'이 쌍으로 이루어져 있어요. 정말 재미있는 논리 게임 같죠?")

# 1. Config 로드 시도 (필수 요구사항)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"\n✅ 성공! 사용 가능한 Config 목록을 불러왔어요: {configs}")
    # 이 데이터셋은 config가 복잡하지 않으니, 그냥 기본 설정으로 진행할게요!
    selected_config = configs[0] if configs else None
except Exception as e:
    print(f"ℹ️ (config 체크) 해당 데이터셋은 별도의 Config가 필요 없는 기본 설정입니다.")
    selected_config = None


dataset = None
dataset_iterator = None

# 2. 데이터 로딩 로직: 스트리밍 우선 시도 (가장 빠르고 효율적인 방법!)
print("\n🚀 2단계: 데이터셋을 스트리밍 모드로 로드해 보겠습니다. (데이터가 클 때 최고!)")

try:
    # streaming=True를 통해 메모리 효율적으로 데이터셋을 탐색합니다.
    dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=True)
    print("🎉 스트리밍 로드 성공! 메모리 부담 없이 데이터를 만질 수 있게 되었어요.")

except Exception as e:
    # 만약 스트리밍 로드가 실패하거나 권한 문제 등이 발생하면 (혹은 너무 느리다면),
    # 소량의 데이터를 다운로드하여 안정적으로 진행합니다.
    print(f"⚠️ (Warning) 스트리밍 로드에 실패하거나 문제가 발생했습니다 ({e}).")
    print("⚡️ 안정성을 위해 소량의 데이터셋을 다운로드하여 진행할게요.")
    try:
        dataset = load_dataset(DATASET_NAME, split=TARGET_SPLIT, streaming=False)
    except Exception as e_fallback:
        print(f"❌ 치명적인 오류: 데이터셋을 로드하는 데 실패했습니다. 오류: {e_fallback}")
        exit()


# 3. 데이터 샘플링 (전체 데이터를 쓰는 대신, 상위 k개만 살펴봅니다!)
print(f"\n💡 3단계: 총 {SAMPLE_COUNT}개의 샘플만 뽑아서 살펴보겠습니다. (시간 절약!)")

if hasattr(dataset, "take"):
    # .take() 메서드가 존재하면 스트리밍 데이터셋(IterableDataset)이므로 이를 사용
    # take()를 통해 처음 N개만 가져옵니다.
    dataset_iterator = dataset.take(SAMPLE_COUNT)
    # 반복자가 되도록 변환합니다.
    sampled_dataset_list = list(dataset_iterator)
else:
    # 일반 Dataset인 경우, list()로 변환하여 사용
    sampled_dataset_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))


# ========================================================================
# ✨ 4단계: 창의적 실습 - 구조화된 프롬프트 엔지니어링
# ========================================================================

print("\n========================================================================")
print(f"✨ 🎨 실습 목표: 원본 데이터(시나리오, 공격/방어 논리)를 AI가 이해하기 쉬운 '프롬프트 카드'로 변환하기")
print("🤖 이 데이터셋은 논리를 '만드는 과정'을 학습하기 좋습니다. 이걸 LLM 프롬프트로 재구성 해볼까요?")
print("========================================================================\n")

# 4. 샘플 반복 및 프롬프트 생성
for i, sample in enumerate(sampled_dataset_list):
    # 핵심 데이터 추출
    scenario = sample.get("scenario_description", "N/A")
    attack_reasoning = sample.get("attack_reasoning", "N/A")
    defense_reasoning = sample.get("defense_reasoning", "N/A")

    print(f"==================== [샘플 {i+1}/{len(sampled_dataset_list)}] ====================")
    
    # 💻 구조화된 프롬프트 템플릿 생성
    prompt_template = f"""
[TASK] 당신은 숙련된 전술 사령관 AI입니다. 주어진 전장 상황(Scenario)을 분석하고,
특정 목표에 따른 최적의 대응 전략을 상세히 추론하세요.

[CONTEXT: 전장 상황 (Scenario)]
---
{scenario}
---

[ACTION REQUIRED: 공격 전략 수립]
공격 부대에게 필요한 전략적 우위와 행동 원칙은 무엇입니까? (Reasoning: {attack_reasoning})

[ADVERSARY COUNTER: 방어 전략 분석]
방어 부대가 이 상황을 어떻게 역이용하여 버틸 수 있습니까? (Reasoning: {defense_reasoning})

[OUTPUT FORMAT]
---
1. 위협 평가: 현재 전장의 가장 큰 위협은 무엇인가?
2. 공격 계획: 가장 효과적인 공격 단계 3가지 (순서대로)를 서술하세요.
3. 방어 예측: 적의 공격에 대응할 수 있는 가장 강력한 방어 전술을 제안하세요.
---
"""
    # 결과를 출력하여, 학습자에게 '이런 식으로 AI에게 질문을 던져야 한다'는 구조를 보여줍니다.
    print(prompt_template)
    
    # 잠깐의 딜레이를 주어, 출력물이 너무 빠르게 넘어가지 않게 합니다.
    time.sleep(0.1)

print("\n========================================================================")
print("✅ 튜터 코멘트: 와우! 정말 멋진 전술가 AI의 밑그림을 완성했어요!")
print("이 코드를 통해, 우리는 데이터를 단순히 읽는 것을 넘어, 데이터의 '구조'를 파악하고")
print("가장 효과적인 '질문(Prompt)' 형태로 재가공하는 방법을 배웠습니다. 이게 바로 AI 개발의 핵심이에요!")
print("🎉 수고하셨습니다! 다음에는 이 프롬프트를 가지고 실제로 LLM을 호출해 보세요!")
print("========================================================================")